In [ ]:
import autogen
import os

config_list = [{"model": "gpt-4-turbo", "api_key": os.environ["OPENAI_API_KEY"]}]
llm_config = {"config_list": config_list}

In [ ]:
import json
import os

import pandas as pd

import chromadb
from typing_extensions import Annotated

# import autogen
# from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent


# Accepted file formats for that can be stored in
# a vector database instance
# from autogen.retrieve_utils import TEXT_FORMATS

In [ ]:
dataset_schema= """
Here's how the data schema looks like:

country: Name of the country (categorical).
continent: Continent to which the country belongs (categorical).
year: Year for which the data is recorded (numeric).
lifeExp: Life expectancy of the population (numeric).
pop: Population of the country (numeric).
gdpPercap: GDP per capita of the country (numeric).
"""

code_problem = """
I need a page with a bar chart shoing the population per continent 
and a scatter chart showing the life expectency per country as a function gdp. 
Make a filter on the country column and use a dropdown as selector. This filter should only 
apply to the bar chart. The bar chart should be a stacked bar chart, while 
the scatter chart should be colored by the column 'continent'. Make another filter 
that filters the GDP column, and it only applies to the scatter chart. I also want 
a table that shows the data. The title of the page should be: `This is big time data`. I also want a second page with two 
cards on it. One should be a card saying: `This was the jolly data dashboard, it was created in Vizro which is amazing` 
, and the second card should refer to the 
 documentation and link to `https://vizro.readthedocs.io/`. The title of the dashboard should be: `My wonderful 
jolly dashboard showing a lot of data`.
"""

#### steps

1. first summarize which components are requested. For each component, summarize all detail specifications.
2. Call corresponding models, e.g., Page, Graph, Filter, Checklist, Card
3. Add RAG for context
4. Summarize all info + context and generate dashboard code
5. Run the code to validate

In [ ]:
dashboard_designer = autogen.AssistantAgent(
    name="dashboard_designer",
    llm_config=llm_config,
    system_message="""
        You are a professional dashboard designer, known for your skills to summarize user requests.
        1. You first summarize which Vizro models are requested. 
        2. For each Vizro model, summarize all detail specifications. i.e., which arguments are needed and their values.

        Available Vizro models to choose from:
        [
         'AgGrid' # for table creation,
         'Button',
         'Card',
         'Filter', # accept Dropdown and RangeSlider
         'Graph',
         'Layout',
         'Page',
         'Parameter',
        ]

        Vizro model metadata:
        {
        "Card": {"title": "Card", "description": "Creates a card utilizing `dcc.Markdown` as title and text component.\\n\\nArgs:\\n    type (Literal[\\"card\\"]): Defaults to `\\"card\\"`.\\n    text (str): Markdown string to create card title/text that should adhere to the CommonMark Spec.\\n    href (str): URL (relative or absolute) to navigate to. If not provided the Card serves as a text card\\n        only. Defaults to `\\"\\"`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "type": {"title": "Type", "default": "card", "enum": ["card"], "type": "string"}, "text": {"title": "Text", "description": "Markdown string to create card title/text that should adhere to the CommonMark Spec.", "type": "string"}, "href": {"title": "Href", "description": "URL (relative or absolute) to navigate to. If not provided the Card serves as a text card only.", "default": "", "type": "string"}}, "required": ["text"], "additionalProperties": false},
        "Graph": {"title": "Graph", "description": "Wrapper for `dcc.Graph` to visualize charts in dashboard.\\n\\nArgs:\\n    type (Literal[\\"graph\\"]): Defaults to `\\"graph\\"`.\\n    figure (CapturedCallable): See [`CapturedCallable`][vizro.models.types.CapturedCallable].\\n    actions (List[Action]): See [`Action`][vizro.models.Action]. Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "type": {"title": "Type", "default": "graph", "enum": ["graph"], "type": "string"}, "actions": {"title": "Actions", "default": [], "type": "array", "items": {"$ref": "#/definitions/Action"}}}, "additionalProperties": false, "definitions": {"Action": {"title": "Action", "description": "Action to be inserted into `actions` of relevant component.\\n\\nArgs:\\n    function (CapturedCallable): See [`CapturedCallable`][vizro.models.types.CapturedCallable].\\n    inputs (List[str]): Inputs in the form `<component_id>.<property>` passed to the action function.\\n        Defaults to `[]`.\\n    outputs (List[str]): Outputs in the form `<component_id>.<property>` changed by the action function.\\n        Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "inputs": {"title": "Inputs", "description": "Inputs in the form `<component_id>.<property>` passed to the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}, "outputs": {"title": "Outputs", "description": "Outputs in the form `<component_id>.<property>` changed by the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}}, "additionalProperties": false}}}
        "Dropdown": {"title": "Dropdown", "description": "Categorical single/multi-option selector `Dropdown`.\\n\\nCan be provided to [`Filter`][vizro.models.Filter] or\\n[`Parameter`][vizro.models.Parameter]. Based on the underlying\\n[`dcc.Dropdown`](https://dash.plotly.com/dash-core-components/dropdown).\\n\\nArgs:\\n    type (Literal[\\"dropdown\\"]): Defaults to `\\"dropdown\\"`.\\n    options (OptionsType): See [`OptionsType`][vizro.models.types.OptionsType]. Defaults to `[]`.\\n    value (Optional[Union[SingleValueType, MultiValueType]]): See\\n        [`SingleValueType`][vizro.models.types.SingleValueType] and\\n        [`MultiValueType`][vizro.models.types.MultiValueType]. Defaults to `None`.\\n    multi (bool): Whether to allow selection of multiple values. Defaults to `True`.\\n    title (str): Title to be displayed. Defaults to `\\"\\"`.\\n    actions (List[Action]): See [`Action`][vizro.models.Action]. Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "type": {"title": "Type", "default": "dropdown", "enum": ["dropdown"], "type": "string"}, "options": {"title": "Options", "default": [], "anyOf": [{"type": "array", "items": {"type": "boolean"}}, {"type": "array", "items": {"type": "number"}}, {"type": "array", "items": {"type": "string"}}, {"type": "array", "items": {"type": "string", "format": "date"}}, {"type": "array", "items": {"$ref": "#/definitions/OptionsDictType"}}]}, "value": {"title": "Value", "anyOf": [{"type": "boolean"}, {"type": "number"}, {"type": "string"}, {"type": "string", "format": "date"}, {"type": "array", "items": {"type": "boolean"}}, {"type": "array", "items": {"type": "number"}}, {"type": "array", "items": {"type": "string"}}, {"type": "array", "items": {"type": "string", "format": "date"}}]}, "multi": {"title": "Multi", "description": "Whether to allow selection of multiple values", "default": true, "type": "boolean"}, "title": {"title": "Title", "description": "Title to be displayed", "default": "", "type": "string"}, "actions": {"title": "Actions", "default": [], "type": "array", "items": {"$ref": "#/definitions/Action"}}}, "additionalProperties": false, "definitions": {"OptionsDictType": {"title": "OptionsDictType", "type": "object", "properties": {"label": {"title": "Label", "type": "string"}, "value": {"title": "Value", "anyOf": [{"type": "boolean"}, {"type": "number"}, {"type": "string"}, {"type": "string", "format": "date"}]}}, "required": ["label", "value"], "additionalProperties": false}, "Action": {"title": "Action", "description": "Action to be inserted into `actions` of relevant component.\\n\\nArgs:\\n    function (CapturedCallable): See [`CapturedCallable`][vizro.models.types.CapturedCallable].\\n    inputs (List[str]): Inputs in the form `<component_id>.<property>` passed to the action function.\\n        Defaults to `[]`.\\n    outputs (List[str]): Outputs in the form `<component_id>.<property>` changed by the action function.\\n        Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "inputs": {"title": "Inputs", "description": "Inputs in the form `<component_id>.<property>` passed to the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}, "outputs": {"title": "Outputs", "description": "Outputs in the form `<component_id>.<property>` changed by the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}}, "additionalProperties": false}}}
        "AgGrid": {"title": "AgGrid", "description": "Wrapper for `dash-ag-grid.AgGrid` to visualize grids in dashboard.\\n\\nArgs:\\n    type (Literal[\\"ag_grid\\"]): Defaults to `\\"ag_grid\\"`.\\n    figure (CapturedCallable): AgGrid like object to be displayed. For more information see:\\n        [`dash-ag-grid.AgGrid`](https://dash.plotly.com/dash-ag-grid).\\n    title (str): Title of the table. Defaults to `\\"\\"`.\\n    actions (List[Action]): See [`Action`][vizro.models.Action]. Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "type": {"title": "Type", "default": "ag_grid", "enum": ["ag_grid"], "type": "string"}, "title": {"title": "Title", "description": "Title of the AgGrid", "default": "", "type": "string"}, "actions": {"title": "Actions", "default": [], "type": "array", "items": {"$ref": "#/definitions/Action"}}}, "additionalProperties": false, "definitions": {"Action": {"title": "Action", "description": "Action to be inserted into `actions` of relevant component.\\n\\nArgs:\\n    function (CapturedCallable): See [`CapturedCallable`][vizro.models.types.CapturedCallable].\\n    inputs (List[str]): Inputs in the form `<component_id>.<property>` passed to the action function.\\n        Defaults to `[]`.\\n    outputs (List[str]): Outputs in the form `<component_id>.<property>` changed by the action function.\\n        Defaults to `[]`.", "type": "object", "properties": {"id": {"title": "Id", "description": "ID to identify model. Must be unique throughout the whole dashboard.When no ID is chosen, ID will be automatically generated.", "default": "", "type": "string"}, "inputs": {"title": "Inputs", "description": "Inputs in the form `<component_id>.<property>` passed to the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}, "outputs": {"title": "Outputs", "description": "Outputs in the form `<component_id>.<property>` changed by the action function.", "default": [], "pattern": "^[^.]+[.][^.]+$", "type": "array", "items": {"type": "string", "pattern": "^[^.]+[.][^.]+$"}}}, "additionalProperties": false}}}
        }
        """,
)

def termination_msg(x):
    return isinstance(x, dict) and "TERMINATE" == str(x.get("content", ""))[-9:].upper()



rag_aid = RetrieveUserProxyAgent(
    name="user_Assistant",
    is_termination_msg=termination_msg,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "code",
        "docs_path": [
                # "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/schemas/0.1.16.json",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/dashboard.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/pages.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/layouts.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/table.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/filters.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/graph.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/card-button.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/navigation.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/parameters.md",
                "https://raw.githubusercontent.com/mckinsey/vizro/main/vizro-core/docs/pages/user-guides/selectors.md",
            ],
        "chunk_token_size": 1000,
        "model": config_list[0]["model"],
        "client": chromadb.PersistentClient(path="/tmp/chromadb"),
        "collection_name": "vizro",
        "get_or_create": True,
    },
    code_execution_config=False,  # we don't want to execute code in this case.
    description="Assistant who has extra content retrieval power for solving difficult problems.",
)


card_expert = autogen.AssistantAgent(
    name="card_expert",
    is_termination_msg=termination_msg,
    system_message="""
    You are a senior python engineer who understands Vizro docs well, you provide python code to answer questions. You always refer to Vizro docs before answer questions.
    If I mentioned <Card>, you provide the code to construct a Vizro Card object.
    Be accurate and succinct, no need to explain your code.
    """,
    llm_config=llm_config,
)

graph_expert = autogen.AssistantAgent(
    name="graph_expert",
    is_termination_msg=termination_msg,
    system_message="""
    You are a senior python engineer who understands Vizro docs well, you provide python code to answer questions. You always refer to Vizro docs about Graph before answer questions.
    If I mentioned <Graph>, you provide the code to construct a Vizro Graph object. Never use seaborn or matplotlib.
    Be accurate and succinct, no need to explain your code.
    """,
    llm_config=llm_config,
)

aggrid_expert = autogen.AssistantAgent(
    name="aggrid_expert",
    is_termination_msg=termination_msg,
    system_message="""
    You are a senior python engineer who understands Vizro docs well, you provide python code to answer questions. You always refer to Vizro docs about table AgGrid before answer questions.
    If I mentioned <table> or <AgGrid>, you provide the code to construct a Vizro AgGrid object. Remember to write `from vizro.tables import dash_ag_grid`
    """,
    llm_config=llm_config,
)

filter_expert = autogen.AssistantAgent(
    name="filter_expert",
    is_termination_msg=termination_msg,
    system_message="""
    You are a senior python engineer who understands Vizro docs well, you provide python code to answer questions. You always refer to Vizro docs before answer questions.
    If I mentioned <Filter>, you provide the code to construct a Vizro Filter object inside dashboard.
    example:
    `
    page = vm.Page(
        components=[vm.Graph()],
        controls=[vm.Filter(column="species"],
    )
    `
    Be accurate and succinct, no need to explain your code.
    """,
    llm_config=llm_config,
)

dashboard_expert = autogen.AssistantAgent(
    name="dashboard_expert",
    is_termination_msg=termination_msg,
    system_message="""
    You are a senior python engineer who understands Vizro docs well, you provide python code to answer questions. You always refer to Vizro docs before answer questions.
    If I mentioned <Dashboard>, you provide the code to construct a Vizro Dashboard. Remember to write `from vizro import Vizro`.
    """,
    llm_config=llm_config,
)



user = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "last_n_messages": 1,
        "work_dir": "tasks",
        "use_docker": False,
    },  # Please set use_docker=True if docker is available to run the generated code. Using docker is safer than running the generated code directly.
)



In [ ]:
def retrieve_content(
    message: Annotated[
        str,
        "Refined message which only keeps the relavant info related to the Vizro model I'm asking and can be used to retrieve content for code generation and question answering.",
    ],
    n_results: Annotated[int, "number of results"] = 2,
) -> str:
    rag_aid.n_results = n_results  # Set the number of results to be retrieved.
    # Check if we need to update the context.
    update_context_case1, update_context_case2 = rag_aid._check_update_context(message)
    if (update_context_case1 or update_context_case2) and rag_aid.update_context:
        rag_aid.problem = message if not hasattr(rag_aid, "problem") else rag_aid.problem
        _, ret_msg = rag_aid._generate_retrieve_user_reply(message)
    else:
        _context = {"problem": message, "n_results": n_results}
        ret_msg = rag_aid.message_generator(rag_aid, None, _context)
    return ret_msg if ret_msg else message

rag_aid.human_input_mode = "NEVER"  # Disable human input for rag_aid since it only retrieves content.

for caller in [card_expert, graph_expert, aggrid_expert, filter_expert, dashboard_expert]:
    d_retrieve_content = caller.register_for_llm(
        description="retrieve content for code generation and question answering.", api_style="function"
    )(retrieve_content)

for executor in [user]:
    executor.register_for_execution()(d_retrieve_content)

card_groupchat = autogen.GroupChat(
    agents=[user, card_expert],
    messages=[],
    max_round=4,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)
graph_groupchat = autogen.GroupChat(
    agents=[user, graph_expert],
    messages=[],
    max_round=4,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)
aggrid_groupchat = autogen.GroupChat(
    agents=[user, aggrid_expert],
    messages=[],
    max_round=4,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)
filter_groupchat = autogen.GroupChat(
    agents=[user, filter_expert],
    messages=[],
    max_round=4,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)
dashboard_groupchat = autogen.GroupChat(
    agents=[user, dashboard_expert],
    messages=[],
    max_round=4,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)

card_manager = autogen.GroupChatManager(
    groupchat=card_groupchat, llm_config={"config_list": config_list, "timeout": 60, "temperature": 0}
)
graph_manager = autogen.GroupChatManager(
    groupchat=graph_groupchat, llm_config={"config_list": config_list, "timeout": 60, "temperature": 0}
)
aggrid_manager = autogen.GroupChatManager(
    groupchat=aggrid_groupchat, llm_config={"config_list": config_list, "timeout": 60, "temperature": 0}
)
filter_manager = autogen.GroupChatManager(
    groupchat=filter_groupchat, llm_config={"config_list": config_list, "timeout": 60, "temperature": 0}
)
dashboard_manager = autogen.GroupChatManager(
    groupchat=dashboard_groupchat, llm_config={"config_list": config_list, "timeout": 60, "temperature": 0}
)


In [ ]:
chat_results = user.initiate_chats(
    [
        {
            "recipient": dashboard_designer,
            "message": code_problem + dataset_schema,
            "clear_history": True,
            "silent": False,
            "max_turns": 1,
            "summary_method": "last_msg",
        },
        {
            "recipient": card_manager,
            "message": "These are my requirements. Only use the portion related to Vizro model <Card> as your context",
            "max_turns": 1,
            "summary_method": "last_msg",
        },
        {
            "recipient": graph_manager,
            "message": "These are my requirements. Only use the portion related to Vizro model <Graph> as your context",
            "max_turns": 1,
            "summary_method": "last_msg",
        },
        {
            "recipient": aggrid_manager,
            "message": "These are my requirements. Only use the portion related to Vizro model <AgGrid> as your context",
            "max_turns": 1,
            "summary_method": "last_msg",
        },
        {
            "recipient": filter_manager,
            "message": "These are my requirements. Only use the portion related to Vizro model <Filter> as your context",
            "max_turns": 1,
            "summary_method": "last_msg",
        },
        {
            "recipient": dashboard_manager,
            "message": "These are my requirements. Only use the portion related to Vizro model <Dashboard> as your context",
            "max_turns": 1,
            "summary_method": "last_msg",
        },
    ]
)